## Modification #1 - Import NiftiLabelsMasker
**Avertissement original :** FutureWarning — `nilearn.input_data` est déprécié depuis la version 0.9
**Changement :** Mise à jour du chemin d'import
**Pourquoi :** Bonne pratique pour compatibilité future

In [1]:
# Install the needed packages
!pip install nilearn
!pip install mne-connectivity
!pip install mlxtend
from nilearn import datasets

#code original
#from nilearn.input_data import NiftiLabelsMasker
from nilearn.maskers import NiftiLabelsMasker
from nilearn.connectome import ConnectivityMeasure
from mne_connectivity.viz import plot_connectivity_circle

import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import mne_connectivity

In [2]:
# import the behavioral dataset
url = "https://github.com/WeiHungLin/BHS_project/raw/main/participants_childrenonly.csv"
df_behav = pd.read_csv(url)
df_behav

,participant_id,Age,AgeGroup,Child_Adult,Gender,Handedness,ToM Booklet-Matched,ToM Booklet-Matched-NOFB,FB_Composite,FB_Group,WPPSI BD raw,WPPSI BD scaled,KBIT_raw,KBIT_standard,DCCS Summary,Scanlog: Scanner,Scanlog: Coil,Scanlog: Voxel slize,Scanlog: Slice Gap
0,sub-pixar001,4.774812,4yo,child,M,R,0.80,0.736842,6,pass,22.0,13.0,NaN,NaN,3.0,3T1,7-8yo 32ch,3mm iso,0.1
1,sub-pixar002,4.856947,4yo,child,F,R,0.72,0.736842,4,inc,18.0,9.0,NaN,NaN,2.0,3T1,7-8yo 32ch,3mm iso,0.1
2,sub-pixar003,4.153320,4yo,child,F,R,0.44,0.421053,3,inc,15.0,9.0,NaN,NaN,3.0,3T1,7-8yo 32ch,3mm iso,0.1
3,sub-pixar004,4.473648,4yo,child,F,R,0.64,0.736842,2,fail,17.0,10.0,NaN,NaN,3.0,3T1,7-8yo 32ch,3mm iso,0.2
4,sub-pixar005,4.837782,4yo,child,F,R,0.60,0.578947,4,inc,13.0,5.0,NaN,NaN,2.0,3T1,7-8yo 32ch,3mm iso,0.2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
117,sub-pixar118,10.500000,8-12yo,child,F,R,1.00,1.000000,6,pass,NaN,NaN,37.0,119.0,NaN,3T1,32ch adult,3.13 mm iso,NaN
118,sub-pixar119,8.620000,8-12yo,child,F,R,1.00,1.000000,5,pass,NaN,NaN,33.0,121.0,NaN,3T1,32ch adult,3.13 mm iso,NaN
119,sub-pixar120,11.480000,8-12yo,child,F,R,1.00,1.000000,5,pass,NaN,NaN,38.0,117.0,NaN,3T1,32ch adult,3.13 mm iso,NaN
120,sub-pixar121,8.760000,8-12yo,child,F,R,1.00,1.000000,6,pass,NaN,NaN,37.0,130.0,NaN,3T1,32ch adult,3.13 mm iso,NaN


In [3]:
participant_df = pd.DataFrame({
    'Number': [122],
    'Age Mean': [6.71],
    'Age Range': ["3 - 12"],
    'ToM Mean': [0.775],
    'ToM Range':["0.24 - 1"]
}, index = [0])


styled_participant_df = participant_df.style.set_properties(**{'background-color': 'lightblue',
                                       'color': 'black',
                                       'font-family': 'Arial',
                                       'text-align': 'center'})


In [4]:
styled_participant_df

,Number,Age Mean,Age Range,ToM Mean,ToM Range
0,122,6.710000,3 - 12,0.775000,0.24 - 1


In [5]:
# fetch the preprocessed rs-fMRI dataset
data = datasets.fetch_development_fmri()

[fetch_development_fmri] Dataset found in /home/nrioux/nilearn_data/development_fmri

[fetch_development_fmri] Dataset found in /home/nrioux/nilearn_data/development_fmri/development_fmri

[fetch_development_fmri] Dataset found in /home/nrioux/nilearn_data/development_fmri/development_fmri

In [6]:
# Check the dataset size
len(data.func)

155

In [7]:
print(df_behav['AgeGroup'].value_counts().sort_index())

AgeGroup
3yo       17
4yo       14
5yo       34
7yo       23
8-12yo    34
Name: count, dtype: int64


#Modification 2: changement d'atlas

## Bug #1 - Réduction des régions
Problème : 400 régions × 122 sujets dépasse la RAM
Solution : Réduire à 100 régions avec n_rois=100

## Bug #2 - Réduction du nombre de participants
Problème: dépasse toujours le RAM même avec 100 rois
Solution: utilisé 50 participants

## Bug #3 - Réduction du nombre de participants
Problème: dépasse toujours le RAM même avec 50 participants
Solution: utilisé 40 participants

In [8]:
from nilearn import datasets

parcellations = datasets.fetch_atlas_schaefer_2018(n_rois=100)
atlas_filename = parcellations.maps

#masker = NiftiLabelsMasker(labels_img = atlas_filename,
#                           standardize = True,
#                           memory = 'nilearn_cache',
#                           verbose = 0)

masker = NiftiLabelsMasker(labels_img=atlas_filename,
                           standardize='zscore_sample',  # ← à jour
                           memory='nilearn_cache',
                           verbose=0)

correlation_measure = ConnectivityMeasure(kind='correlation',
                      vectorize=True,
                      discard_diagonal=True,
                      standardize='zscore_sample')

[fetch_atlas_schaefer_2018] Dataset found in /home/nrioux/nilearn_data/schaefer_2018

In [9]:
# Create the correlation matrix for all children
all_features = []

for i,sub in enumerate(data.func):
  if i > 32 :  # 40 enfants seulement
    # Extract the timeseries from the ROIs in the atlas
    time_series = masker.fit_transform(sub, confounds = data.confounds[i])
    # Create a correlation matrix
    correlation_matrix = correlation_measure.fit_transform([time_series])[0]
    # Add to the list
    all_features.append(correlation_matrix)
    # Kepp track of the progress
    print('finished %s of %s'%((i+1),len(data.func)))
  else:
    continue

finished 34 of 155


finished 35 of 155


finished 36 of 155


finished 37 of 155


finished 38 of 155


...

finished 70 of 155


finished 71 of 155


finished 72 of 155


finished 73 of 155


finished 74 of 155


In [ ]:
# Create the X variables
X_features = np.array(all_features)
X_features.shape

In [ ]:
plt.imshow(X_features, aspect = 'auto', cmap="turbo")
plt.colorbar()
plt.title('feature matrix')
plt.xlabel('features')
plt.ylabel('subjects')

# Bug 4:
le nombre de participants en x et en y ne correspond plus
Solution: garder seulement les 37 premiers

In [ ]:
# Look over the variable in the behavioral dataframe
df_behav.columns

In [ ]:
# Garder juste les 40 mêmes sujets
df_behav_subset = df_behav.copy()

In [ ]:
# Assign the ToM score as the y variable
y_ToM = df_behav_subset["ToM Booklet-Matched"]

In [ ]:
y_ToM

In [ ]:
# Take a look at the distribution of the ToM (target variable)
sns.histplot(y_ToM, kde = True, stat = "probability", kde_kws = dict(cut=3))

In [ ]:
# Assign groups to subjects based on the ToM scores

# Create a function to assign subjects to different groups based on their ToM scores
def assign_group(score):
    if score >= 0.2 and score < 0.4:
        return 'Group 1'
    elif score >= 0.4 and score < 0.6:
        return 'Group 2'
    elif score >= 0.6 and score < 0.8:
        return 'Group 3'
    elif score >= 0.8 and score <= 1.0:
        return 'Group 4'
    else:
        return 'Unknown'

# Add a new column 'Group' based on the 'ToM_scores' column
df_behav_subset["ToMGroup"] = df_behav_subset["ToM Booklet-Matched"].apply(assign_group)

In [ ]:
ToM_class = df_behav_subset["ToMGroup"]
ToM_class.value_counts()

In [ ]:
# Fit Machine Learning to our data

from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_predict, cross_val_score
from sklearn.metrics import r2_score
from sklearn.metrics import explained_variance_score
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import median_absolute_error
from sklearn.svm import SVR

l_svr = SVR(kernel="linear")


X_train, X_val, y_train, y_val = train_test_split(
    X_features, # x
    y_ToM, # y
    test_size = 0.3,
    shuffle = True,  # shuffle dataset before splitting
    stratify = ToM_class, # keep distribution of ToMclass consistent b/t train and test sets
    random_state = 123
)

print('training: ', len(X_train),
   'testing: ', len(X_val))

In [ ]:
# Visualized the distributions to be sure they are matched b/t train and test subset
import seaborn as sns

sns.histplot(y_train, label='train', kde = True, stat = "probability", kde_kws=dict(cut=3), color = 'orange')
sns.histplot(y_val, label='test', kde=True, stat='probability', kde_kws=dict(cut=3))
plt.legend()

In [ ]:
l_svr = SVR(kernel="linear")

# k-fold cross validation predict
l_svr.fit(X_train, y_train)
y_pred = cross_val_predict(l_svr, X_train, y_train, cv=10)
acc = cross_val_score(l_svr, X_train, y_train, cv=10)
expvar = cross_val_score(l_svr, X_train, y_train, cv=10, scoring='explained_variance')
mae = cross_val_score(l_svr, X_train, y_train, cv=10, scoring='neg_mean_absolute_error')
medae = cross_val_score(l_svr, X_train, y_train, cv=10, scoring='neg_median_absolute_error')

# Métriques validation croisée
overall_acc_1 = r2_score(y_train, y_pred)
overall_expvar_1 = explained_variance_score(y_train, y_pred)
overall_mae_1 = mean_absolute_error(y_train, y_pred)
overall_medae_1 = median_absolute_error(y_train, y_pred)

results_df = pd.DataFrame({
    'R2': [overall_acc_1],
    'Explained Variance': [overall_expvar_1],
    'Mean Absolute Error': [overall_mae_1],
    'Median Absolute Error': [overall_medae_1]
})
V1_styled_results_df = results_df.style.set_properties(**{
    'background-color': 'lightblue',
    'color': 'black',
    'font-family': 'Arial',
    'text-align': 'center'
})

In [ ]:
V1_styled_results_df

In [ ]:
# Évaluation sur le test set
y_val_pred = l_svr.predict(X_val)
test_r2 = r2_score(y_val, y_val_pred)
test_expvar = explained_variance_score(y_val, y_val_pred)
test_mae = mean_absolute_error(y_val, y_val_pred)
test_medae = median_absolute_error(y_val, y_val_pred)

results_df_1 = pd.DataFrame({
    'R2': [test_r2],
    'Explained Variance': [test_expvar],
    'Mean Absolute Error': [test_mae],
    'Median Absolute Error': [test_medae]
})
styled_results_df_1 = results_df_1.style.set_properties(**{
    'background-color': 'lightblue',
    'color': 'black',
    'font-family': 'Arial',
    'text-align': 'center'
})

In [ ]:
styled_results_df_1

In [ ]:
# plot result
sns.regplot(x=y_val_pred, y=y_val, marker = 'o')
plt.xlabel('Predicted ToM score')

In [ ]:
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('pca', PCA()),
    ('svr', SVR(kernel='linear'))
])

param_grid = {
    'pca__n_components': [10, 20, 50],
    'svr__C': [ 0.01, 0.1, 1, 10]
}

grid_search = GridSearchCV(
    pipeline,
    param_grid,
    cv=10,
    scoring='r2',
    verbose=1
)

grid_search.fit(X_train, y_train)

print(f"Meilleurs paramètres: {grid_search.best_params_}")
print(f"Meilleur R2 CV: {grid_search.best_score_:.3f}")

In [ ]:
best_model = grid_search.best_estimator_
y_pred_test = best_model.predict(X_val)
print(f"R2 test: {r2_score(y_val, y_pred_test):.3f}")
print(f"MAE test: {mean_absolute_error(y_val, y_pred_test):.3f}")

In [ ]:
from sklearn.feature_selection import SelectKBest, f_regression

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('selector', SelectKBest(f_regression)),
    ('svr', SVR(kernel='linear'))
])

param_grid = {
    'selector__k': [10, 20, 50],
    'svr__C': [0.01, 0.1, 1, 10]
}

grid_search_kbest = GridSearchCV(
    pipeline,
    param_grid,
    cv=10,
    scoring='r2',
    verbose=1
)

grid_search_kbest.fit(X_train, y_train)

print(f"Meilleurs paramètres: {grid_search_kbest.best_params_}")
print(f"Meilleur R2 CV: {grid_search_kbest.best_score_:.3f}")

In [ ]:
best_model = grid_search_kbest.best_estimator_
y_pred_test = best_model.predict(X_val)
print(f"R2 test: {r2_score(y_val, y_pred_test):.3f}")
print(f"MAE test: {mean_absolute_error(y_val, y_pred_test):.3f}")

hmmm kbest rend le modèle encore plus poche

Étant donné que le r2 est très petit on va modifier certain truc les hypers parametres améliore mais comme pas tant

In [ ]:
print(y_train.describe())
plt.hist(y_train, bins=20)
plt.title('Distribution de y_ToM')
plt.show()


In [ ]:
y_transformed = np.log(y_train + 1)

In [ ]:
plt.hist(y_transformed, bins=20)
plt.title('Distribution y_ToM après log transform')
plt.show()

In [ ]:
# Option 1 : racine carrée (moins agressive que log)
y_sqrt = np.sqrt(y_train)

plt.hist(y_sqrt, bins=20)
plt.title('Distribution y_ToM après racine carré')
plt.show()

In [ ]:
# Refaire le grid search avec y transformé
grid_search = GridSearchCV(
    Pipeline([
        ('scaler', StandardScaler()),
        ('pca', PCA()),
        ('svr', SVR(kernel='linear'))
    ]),
    {
        'pca__n_components': [10, 20, 50],
        'svr__C': [0.01, 0.1, 1, 10]
    },
    cv=10,
    scoring='r2',
    verbose=1
)

grid_search.fit(X_train, y_transformed)

print(f"Meilleurs paramètres: {grid_search.best_params_}")
print(f"Meilleur R2 CV: {grid_search.best_score_:.3f}")

In [ ]:
# Refaire le grid search avec y transformé
grid_search = GridSearchCV(
    Pipeline([
        ('scaler', StandardScaler()),
        ('pca', PCA()),
        ('svr', SVR(kernel='linear'))
    ]),
    {
        'pca__n_components': [10, 20, 50],
        'svr__C': [0.01, 0.1, 1, 10]
    },
    cv=10,
    scoring='r2',
    verbose=1
)

grid_search.fit(X_train, y_sqrt)

print(f"Meilleurs paramètres: {grid_search.best_params_}")
print(f"Meilleur R2 CV: {grid_search.best_score_:.3f}")

Faire seulement 2 groupe de TOM + tout le cerveau

In [ ]:
# Assign groups to subjects based on the ToM scores

# Create a function to assign subjects to different groups based on their ToM scores
median_score_2 = df_behav_subset["ToM Booklet-Matched"].median()

def assign_group_2(score):
    if score >= median_score_2:
        return 'High ToM'
    else:
        return 'Low ToM'

# Add a new column 'Group' based on the 'ToM_scores' column
df_behav_subset["ToMGroup_2"] = df_behav_subset["ToM Booklet-Matched"].apply(assign_group_2)

In [ ]:
ToM_class_2 = df_behav_subset["ToMGroup_2"]
ToM_class_2.value_counts()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_predict, cross_val_score
from sklearn.metrics import r2_score
from sklearn.metrics import explained_variance_score
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import median_absolute_error
from sklearn.svm import SVR

l_svr_2 = SVR(kernel="linear")


X_train_2, X_val_2, y_train_2, y_val_2 = train_test_split(
    X_features, # x
    y_ToM, # y
    test_size = 0.3,
    shuffle = True,  # shuffle dataset before splitting
    stratify = ToM_class_2, # keep distribution of ToMclass consistent b/t train and test sets
    random_state = 123
)

print('training: ', len(X_train),
   'testing: ', len(X_val))

In [ ]:
# Visualized the distributions to be sure they are matched b/t train and test subset

sns.histplot(y_train, label='train', kde = True, stat = "probability", kde_kws=dict(cut=3), color = 'orange')
sns.histplot(y_val, label='test', kde=True, stat='probability', kde_kws=dict(cut=3))
plt.legend()

In [ ]:
# k-fold cross validation predict

l_svr_2.fit(X_train_2, y_train_2)

y_pred_2 = cross_val_predict(l_svr_2, X_train_2, y_train_2, cv = 10)

acc = cross_val_score(l_svr_2, X_train_2, y_train_2, cv = 10)
expvar = cross_val_score(l_svr_2, X_train_2, y_train_2, cv = 10, scoring = 'explained_variance')
mae = cross_val_score(l_svr_2, X_train_2, y_train_2, cv = 10, scoring = 'neg_mean_absolute_error')
medae = cross_val_score(l_svr_2, X_train_2, y_train_2, cv = 10, scoring = 'neg_median_absolute_error')

# The model performance with training dataset
overall_acc_2 = r2_score(y_train_2, y_pred_2)
overall_expvar_2 = explained_variance_score(y_train_2, y_pred_2)
overall_mae_2 = mean_absolute_error(y_train_2, y_pred_2)
overall_medae_2 = median_absolute_error(y_train_2, y_pred_2)


# Create a DataFrame to store the results
results_df = pd.DataFrame({
    'R2': [overall_acc_2],
    'Explained Variance': [overall_expvar_2],
    'Mean Absolute Error': [overall_mae_2],
    'Median Absolute Error': [overall_medae_2]
})


V2_styled_results_df = results_df.style.set_properties(**{'background-color': 'lightblue',
                                       'color': 'black',
                                       'font-family': 'Arial',
                                       'text-align': 'center'})

V2_styled_results_df

In [ ]:
#Evaluate the model with validation dataset
y_val_pred_2 = l_svr_2.predict(X_val_2) # classify ToM class using testing data
acc_2 = l_svr_2.score(X_val_2, y_val_2) # get accuracy (r2)
expvar_2 = explained_variance_score(y_val_2, y_val_pred_2) # get the explained variance
mae_2 = mean_absolute_error(y_val_2, y_val_pred_2) # get mean absolute error
medae_2 = median_absolute_error(y_val_2, y_val_pred_2) # get median absolute error

print('Prediction accuracy (R2): ', acc_2)
print('Explained variance: ', expvar_2)
print('Mean absolute error: ', mae_2)
print('Median absolute error: ', medae_2)

# Create a DataFrame to store the results
results_df_2 = pd.DataFrame({
    'Predictin accuracy (R2)': [acc_2],
    'Explained variance': [expvar_2],
    'Mean absolute error': [mae_2],
    'Median absolute error': [medae_2]
})

V2_styled_results_df_2 = results_df_2.style.set_properties(**{'background-color': 'lightblue',
                                       'color': 'black',
                                       'font-family': 'Arial',
                                       'text-align': 'center'})

In [ ]:
V2_styled_results_df_2

In [ ]:
# Refaire le grid search avec y transformé
grid_search = GridSearchCV(
    Pipeline([
        ('scaler', StandardScaler()),
        ('pca', PCA()),
        ('svr', SVR(kernel='linear'))
    ]),
    {
        'pca__n_components': [10, 20, 50],
        'svr__C': [0.01, 0.1, 1, 10]
    },
    cv=10,
    scoring='r2',
    verbose=1
)

grid_search.fit(X_train_2, y_train_2)

print(f"Meilleurs paramètres: {grid_search.best_params_}")
print(f"Meilleur R2 CV: {grid_search.best_score_:.3f}")

In [ ]:
best_model = grid_search.best_estimator_
y_pred_test = best_model.predict(X_val_2)
print(f"R2 test: {r2_score(y_val, y_pred_test):.3f}")
print(f"MAE test: {mean_absolute_error(y_val, y_pred_test):.3f}")

encore très poche

essayons juste mode par default

In [ ]:
print(masker)
print(masker.labels_)

In [ ]:
default = datasets.fetch_atlas_schaefer_2018(n_rois=100, yeo_networks=7)
labels = default.labels
print(labels)

In [ ]:
# Trouver les indices des ROIs du DMN
dmn_indices = [i for i, label in enumerate(labels) if 'Default' in label]

print(f"Nombre de ROIs DMN: {len(dmn_indices)}")
print("ROIs DMN:")
for i in dmn_indices:
    print(i, labels[i])

In [ ]:
len(dmn_indices)

# Bug: too many indices for array: array is 1-dimensional, but 2 were indexed
solution: nouveau masker sans vectorisé

In [ ]:
print(type(all_features[0]))
print(all_features[0].shape)
print(correlation_measure)

In [ ]:
rows, cols = np.triu_indices(100, k=1)
dmn_vector_indices = [i for i, (r, c) in enumerate(zip(rows, cols))
                      if r in dmn_indices and c in dmn_indices]

print(len(dmn_vector_indices))
print(dmn_indices)

# le code devrait donné 276 (24*24-24)/2 mais me donne 253

In [ ]:
dmn_indices = np.array(dmn_indices) - 1
print(dmn_indices)

In [ ]:
rows, cols = np.triu_indices(100, k=1)
dmn_vector_indices = [i for i, (r, c) in enumerate(zip(rows, cols))
                      if r in dmn_indices and c in dmn_indices]

print(len(dmn_vector_indices))

In [ ]:
X_features_dmn = np.array([sub[dmn_vector_indices] for sub in all_features])
print(X_features_dmn.shape)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_predict, cross_val_score
from sklearn.metrics import r2_score
from sklearn.metrics import explained_variance_score
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import median_absolute_error
from sklearn.svm import SVR

l_svr_dmn = SVR(kernel="linear")


X_train_dmn, X_val_dmn, y_train_dmn, y_val_dmn = train_test_split(
    X_features_dmn, # x
    y_ToM, # y
    test_size = 0.3,
    shuffle = True,  # shuffle dataset before splitting
    stratify = ToM_class_2, # keep distribution of ToMclass consistent b/t train and test sets
    random_state = 123
)

print('training: ', len(X_train),
   'testing: ', len(X_val))

In [ ]:
# Visualized the distributions to be sure they are matched b/t train and test subset

sns.histplot(y_train, label='train', kde = True, stat = "probability", kde_kws=dict(cut=3), color = 'orange')
sns.histplot(y_val, label='test', kde=True, stat='probability', kde_kws=dict(cut=3))
plt.legend()

In [ ]:
# k-fold cross validation predict

l_svr_dmn.fit(X_train_dmn, y_train_dmn)

y_pred_dmn = cross_val_predict(l_svr_dmn, X_train_dmn, y_train_dmn, cv = 10)

acc_dmn = cross_val_score(l_svr_dmn, X_train_dmn, y_train_dmn, cv = 10)
expvar_dmn = cross_val_score(l_svr_dmn, X_train_dmn, y_train_dmn, cv = 10, scoring = 'explained_variance')
mae_dmn = cross_val_score(l_svr_dmn, X_train_dmn, y_train_dmn, cv = 10, scoring = 'neg_mean_absolute_error')
medae_dmn = cross_val_score(l_svr_dmn, X_train_dmn, y_train_dmn, cv = 10, scoring = 'neg_median_absolute_error')

# The model performance with training dataset
overall_acc_dmn = r2_score(y_train_dmn, y_pred_dmn)
overall_expvar_dmn = explained_variance_score(y_train_dmn, y_pred_dmn)
overall_mae_dmn = mean_absolute_error(y_train_dmn, y_pred_dmn)
overall_medae_dmn = median_absolute_error(y_train_dmn, y_pred_dmn)


# Create a DataFrame to store the results
results_df = pd.DataFrame({
    'R2': [overall_acc_dmn],
    'Explained Variance': [overall_expvar_dmn],
    'Mean Absolute Error': [overall_mae_dmn],
    'Median Absolute Error': [overall_medae_dmn]
})


V3_styled_results_df = results_df.style.set_properties(**{'background-color': 'lightblue',
                                       'color': 'black',
                                       'font-family': 'Arial',
                                       'text-align': 'center'})

In [ ]:
V3_styled_results_df

In [ ]:
#Evaluate the model with validation dataset
y_val_pred_dmn = l_svr_dmn.predict(X_val_dmn) # classify ToM class using testing data
acc_dmn = l_svr_dmn.score(X_val_dmn, y_val_dmn) # get accuracy (r2)
expvar_dmn = explained_variance_score(y_val_dmn, y_val_pred_dmn) # get the explained variance
mae_dmn = mean_absolute_error(y_val_dmn, y_val_pred_dmn) # get mean absolute error
medae_dmn = median_absolute_error(y_val_dmn, y_val_pred_dmn) # get median absolute error

print('Prediction accuracy (R2): ', acc_dmn)
print('Explained variance: ', expvar_dmn)
print('Mean absolute error: ', mae_dmn)
print('Median absolute error: ', medae_dmn)

# Create a DataFrame to store the results
results_df_3 = pd.DataFrame({
    'Predictin accuracy (R2)': [acc_dmn],
    'Explained variance': [expvar_dmn],
    'Mean absolute error': [mae_dmn],
    'Median absolute error': [medae_dmn]
})

V3_styled_results_df_3 = results_df_3.style.set_properties(**{'background-color': 'lightblue',
                                       'color': 'black',
                                       'font-family': 'Arial',
                                       'text-align': 'center'})

très étrange c'est moins bon avec le dmn, peu importe je fais quoi ca va etre poche

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from mne_connectivity.viz import plot_connectivity_circle

In [ ]:
from IPython.display import display

display(styled_results_df_1)
display(V2_styled_results_df_2)
display(V3_styled_results_df_3)
#display(V5_styled_results_df_5)

nouvelle idée je vais créer un réseau theorie de l'esprit

In [ ]:
node_names = [l.decode() if isinstance(l, bytes) else l
              for l in parcellations.labels]
node_names = node_names[1:]  # enlever Background

In [ ]:
correlation_measure_tom = ConnectivityMeasure(
    kind='correlation',
    vectorize=False,
    standardize='zscore_sample'
)

correlation_matrix_tom = correlation_measure_tom.fit_transform([time_series])[0]

In [ ]:
correlation_matrix_2 = correlation_matrix_tom.copy()
np.fill_diagonal(correlation_matrix_2, 0)
print(np.min(correlation_matrix_2), np.max(correlation_matrix_2))

In [ ]:
# Nœuds d'intérêt spécifiques
roi_names_of_interest = [
    # TPJ
    '7Networks_RH_Default_Par_1',
    '7Networks_LH_Default_Par_1',
    '7Networks_LH_Default_Par_2',
    # PCC
    '7Networks_LH_Default_pCunPCC_1',
    '7Networks_LH_Default_pCunPCC_2',
    '7Networks_RH_Default_pCunPCC_1',
    '7Networks_RH_Default_pCunPCC_2',
    # dmPFC
    '7Networks_LH_Default_PFC_5',
    '7Networks_LH_Default_PFC_6',
    '7Networks_RH_Default_PFCdPFCm_1',
    '7Networks_RH_Default_PFCdPFCm_2',
    '7Networks_RH_Default_PFCdPFCm_3',
    # vmPFC
    '7Networks_LH_Default_PFC_1',
    '7Networks_LH_Default_PFC_2',
    '7Networks_RH_Default_PFCv_1',
    '7Networks_RH_Default_PFCv_2',
]

roi_indices = [i for i, name in enumerate(node_names) if name in roi_names_of_interest]
roi_names   = [node_names[i]  for i in roi_indices]
#roi_colors  = [node_colors[i] for i in roi_indices]
roi_matrix  = correlation_matrix_2[np.ix_(roi_indices, roi_indices)]
roi_names_short = ['_'.join(name.split('_')[3:]) for name in roi_names]

rows, cols = np.triu_indices(100, k=1)
tom_vector_indices = [i for i, (r, c) in enumerate(zip(rows, cols))
                      if r in roi_indices and c in roi_indices]

print(len(tom_vector_indices))
print(roi_indices)

X_features_tom = np.array([sub[tom_vector_indices] for sub in all_features])
print(X_features_tom.shape)

l_svr_tom = SVR(kernel="linear")

X_train_class_tom, X_val_class_tom, y_train_class_tom, y_val_class_tom = train_test_split(
    X_features_tom, # x
    y_ToM, # y
    test_size = 0.3,
    shuffle = True,  # shuffle dataset before splitting
    stratify = ToM_class_2, # keep distribution of ToMclass consistent b/t train and test sets
    random_state = 123
)

print('training: ', len(X_train_class_tom),
   'testing: ', len(X_val_class_tom))

In [ ]:
# k-fold cross validation predict

l_svr_tom.fit(X_train_class_tom, y_train_class_tom)

y_pred = cross_val_predict(l_svr_tom, X_train_class_tom, y_train_class_tom, cv = 10)

acc_tom = cross_val_score(l_svr_tom, X_train_class_tom, y_train_class_tom, cv = 10)
expvar_tom = cross_val_score(l_svr_tom, X_train_class_tom, y_train_class_tom, cv = 10, scoring = 'explained_variance')
mae_tom = cross_val_score(l_svr_tom, X_train_class_tom, y_train_class_tom, cv = 10, scoring = 'neg_mean_absolute_error')
medae_tom = cross_val_score(l_svr_tom, X_train_class_tom, y_train_class_tom, cv = 10, scoring = 'neg_median_absolute_error')

# The model performance with training dataset
overall_acc_tom = r2_score(y_train_class_tom, y_pred)
overall_expvar_tom = explained_variance_score(y_train_class_tom, y_pred)
overall_mae_tom = mean_absolute_error(y_train_class_tom, y_pred)
overall_medae_tom = median_absolute_error(y_train_class_tom, y_pred)


# Create a DataFrame to store the results
results_df_tom = pd.DataFrame({
    'R2': [overall_acc_tom],
    'Explained Variance': [overall_expvar_tom],
    'Mean Absolute Error': [overall_mae_tom],
    'Median Absolute Error': [overall_medae_tom]
})


V5_styled_results_df = results_df_tom.style.set_properties(**{'background-color': 'lightblue',
                                       'color': 'black',
                                       'font-family': 'Arial',
                                       'text-align': 'center'})

V5_styled_results_df

In [ ]:
#Evaluate the model with validation dataset
y_val_pred = l_svr_tom.predict(X_val_class_tom) # classify ToM class using testing data
acc = l_svr_tom.score(X_val_class_tom, y_val_class_tom) # get accuracy (r2)
expvar = explained_variance_score(y_val_class_tom, y_val_pred) # get the explained variance
mae = mean_absolute_error(y_val_class_tom, y_val_pred) # get mean absolute error
medae = median_absolute_error(y_val_class_tom, y_val_pred) # get median absolute error

print('Prediction accuracy (R2): ', acc)
print('Explained variance: ', expvar)
print('Mean absolute error: ', mae)
print('Median absolute error: ', medae)

# Create a DataFrame to store the results
results_df_5 = pd.DataFrame({
    'Predictin accuracy (R2)': [acc],
    'Explained variance': [expvar],
    'Mean absolute error': [mae],
    'Median absolute error': [medae]
})

V5_styled_results_df_5 = results_df_5.style.set_properties(**{'background-color': 'lightblue',
                                       'color': 'black',
                                       'font-family': 'Arial',
                                       'text-align': 'center'})

vraiment nul même avec les hyper paramètres pas grand chose vont ce passé

regardons un exemple de graphique avec notre le modèle 2 groupe + tout le cerveau

# c'est très poche donc je vais essayer de classifier

/home/nrioux/miniconda3/lib/python3.13/site-packages/sklearn/model_selection/_split.py:813: UserWarning: The least populated class in y has only 4 members, which is less than n_splits=5.
  warnings.warn so je dois prendre juste low high

In [ ]:
y_ToM_class = ToM_class_2

In [ ]:
from sklearn.svm import SVC

X_class_train, X_class_val, y_class_train, y_class_val = train_test_split(
    X_features, # x
    y_ToM_class, # y
    test_size = 0.3,
    shuffle = True,  # shuffle dataset before splitting
    stratify = ToM_class_2, # keep distribution of ToMclass consistent b/t train and test sets
    random_state = 123
)

print('training: ', len(X_train),
   'testing: ', len(X_val))

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import accuracy_score, balanced_accuracy_score

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('svc', SVC(kernel='linear'))
])

y_pred = cross_val_predict(pipeline, X_class_train, y_class_train, cv=5)

print(f"Accuracy: {accuracy_score(y_class_train, y_pred):.3f}")
print(f"Balanced Accuracy: {balanced_accuracy_score(y_class_train, y_pred):.3f}")

In [ ]:
# Étape 2 : entraîner sur TOUT le train, évaluer sur le test (performance finale)
pipeline.fit(X_class_train, y_class_train)
y_class_pred_test = pipeline.predict(X_class_val)
print("\n=== Évaluation finale sur test set (37 sujets) ===")
print(f"Accuracy: {accuracy_score(y_class_val, y_class_pred_test):.3f}")
print(f"Balanced Accuracy: {balanced_accuracy_score(y_class_val, y_class_pred_test):.3f}")

essayons tout de même de voir si le dmn est bon pour classifier

In [ ]:
from sklearn.metrics import confusion_matrix

confusion_matrix_tom = confusion_matrix(y_true=y_class_train, y_pred = y_pred)

cmdf = pd.DataFrame(confusion_matrix_tom, index = ['Faible ToM',"Haut ToM"], columns = ['Faible ToM','Haut ToM'])
sns.heatmap(cmdf, cmap='Reds')

In [ ]:
from sklearn.svm import SVC

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('pca', PCA()),
    ('svc', SVC(kernel='linear'))
])

param_grid = {
    'pca__n_components': [10, 20, 50],
    'svc__C': [0.001, 0.01, 0.1, 1, 10]
}

grid_search = GridSearchCV(
    pipeline,
    param_grid,
    cv=10,
    scoring='balanced_accuracy',
    verbose=1
)

grid_search.fit(X_class_train, y_class_train)

print(f"Meilleurs paramètres: {grid_search.best_params_}")
print(f"Meilleure balanced accuracy CV: {grid_search.best_score_:.3f}")

# Évaluation finale sur test
best_model = grid_search.best_estimator_
y_pred_test = best_model.predict(X_class_val)
print(f"\nBalanced accuracy test: {balanced_accuracy_score(y_class_val, y_pred_test):.3f}")

ca améliore!!!

faire avec dmn

In [ ]:
from sklearn.svm import SVC

X_train_class_dmn, X_val_class_dmn, y_train_class_dmn, y_val_class_dmn = train_test_split(
    X_features_dmn, # x
    y_ToM_class, # y
    test_size = 0.3,
    shuffle = True,  # shuffle dataset before splitting
    stratify = ToM_class_2, # keep distribution of ToMclass consistent b/t train and test sets
    random_state = 123
)

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import accuracy_score, balanced_accuracy_score

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('svc', SVC(kernel='linear'))
])

y_pred_dmn = cross_val_predict(pipeline, X_train_class_dmn, y_train_class_dmn, cv=5)

print(f"Accuracy: {accuracy_score(y_train_class_dmn, y_pred_dmn):.3f}")
print(f"Balanced Accuracy: {balanced_accuracy_score(y_train_class_dmn, y_pred_dmn):.3f}")

In [ ]:
# Étape 2 : entraîner sur TOUT le train, évaluer sur le test (performance finale)
pipeline.fit(X_train_class_dmn, y_train_class_dmn)
y_pred_test_dmn = pipeline.predict(X_val_class_dmn)
print("\n=== Évaluation finale sur test set (37 sujets) ===")
print(f"Accuracy: {accuracy_score(y_val_class_dmn, y_pred_test_dmn):.3f}")
print(f"Balanced Accuracy: {balanced_accuracy_score(y_val_class_dmn, y_pred_test_dmn):.3f}")

ok pas tres grosses différence ici

In [ ]:
confusion_matrix_dmn = confusion_matrix(y_true=y_train_class_dmn, y_pred = y_pred)

cmdf = pd.DataFrame(confusion_matrix_dmn, index = ['Faible ToM',"Haut ToM"], columns = ['Faible ToM','Haut ToM'])
sns.heatmap(cmdf, cmap='Purples')

In [ ]:
from sklearn.svm import SVC

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('pca', PCA()),
    ('svc', SVC(kernel='linear'))
])

param_grid = {
    'pca__n_components': [10, 20, 50],
    'svc__C': [0.001, 0.01, 0.1, 1, 10]
}

grid_search = GridSearchCV(
    pipeline,
    param_grid,
    cv=10,
    scoring='balanced_accuracy',
    verbose=1
)

grid_search.fit(X_train_class_dmn, y_train_class_dmn)

print(f"Meilleurs paramètres: {grid_search.best_params_}")
print(f"Meilleure balanced accuracy CV: {grid_search.best_score_:.3f}")

# Évaluation finale sur test
best_model = grid_search.best_estimator_
y_pred_test = best_model.predict(X_val_class_dmn)
print(f"\nBalanced accuracy test: {balanced_accuracy_score(y_val_class_dmn, y_pred_test):.3f}")

ok je vais tester mon tom network

In [ ]:
X_train_class_tom, X_val_class_tom, y_train_class_tom, y_val_class_tom = train_test_split(
    X_features_tom, # x
    y_ToM_class, # y
    test_size = 0.3,
    shuffle = True,  # shuffle dataset before splitting
    stratify = ToM_class_2, # keep distribution of ToMclass consistent b/t train and test sets
    random_state = 123
)

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('svc', SVC(kernel='linear'))
])

y_pred_tom = cross_val_predict(pipeline, X_train_class_tom, y_train_class_tom, cv=5)

print(f"Accuracy: {accuracy_score(y_train_class_tom, y_pred_tom):.3f}")
print(f"Balanced Accuracy: {balanced_accuracy_score(y_train_class_tom, y_pred_tom):.3f}")

# Étape 2 : entraîner sur TOUT le train, évaluer sur le test (performance finale)
pipeline.fit(X_train_class_tom, y_train_class_tom)
y_pred_test_tom = pipeline.predict(X_val_class_tom)
print("\n=== Évaluation finale sur test set (37 sujets) ===")
print(f"Accuracy: {accuracy_score(y_val_class_tom, y_pred_test_tom):.3f}")
print(f"Balanced Accuracy: {balanced_accuracy_score(y_val_class_tom, y_pred_test_tom):.3f}")

In [ ]:
confusion_matrix_tom = confusion_matrix(y_true=y_train_class_tom, y_pred = y_pred_tom)

cmdf = pd.DataFrame(confusion_matrix_tom, index = ['Faible ToM',"Haut ToM"], columns = ['Faible ToM','Haut ToM'])
sns.heatmap(cmdf, cmap='Blues')

In [ ]:
from sklearn.svm import SVC

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('pca', PCA()),
    ('svc', SVC(kernel='linear'))
])

param_grid = {
    'pca__n_components': [10, 20, 50],
    'svc__C': [0.001, 0.01, 0.1, 1, 10]
}

grid_search = GridSearchCV(
    pipeline,
    param_grid,
    cv=10,
    scoring='balanced_accuracy',
    verbose=1
)

grid_search.fit(X_train_class_tom, y_train_class_tom)

print(f"Meilleurs paramètres: {grid_search.best_params_}")
print(f"Meilleure balanced accuracy CV: {grid_search.best_score_:.3f}")

# Évaluation finale sur test
best_model = grid_search.best_estimator_
y_pred_test = best_model.predict(X_val_class_tom)
print(f"\nBalanced accuracy test: {balanced_accuracy_score(y_val_class_tom, y_pred_test):.3f}")

https://www.maartengrootendorst.com/blog/validate/

In [ ]:
#1er modèle

In [ ]:
from scipy.stats import wilcoxon

scores_modele1 = np.array([0.56428571, 0.60714286, 0.52083333, 0.45138889, 0.58333333])
scores_modele2 = np.array([0.29285714, 0.48571429, 0.64583333, 0.45138889, 0.46527778])

stat, p_value = wilcoxon(scores_modele1, scores_modele2)
print(f"Statistique : {stat:.3f}")
print(f"p-value : {p_value:.3f}")

In [ ]:
# Access the feature importances used by the model
l_svr.coef_

print("Maximum value:", np.max(l_svr.coef_))
print("Minimum value:", np.min(l_svr.coef_))

# Plot the weights to see the distribution
plt.bar(range(l_svr.coef_.shape[-1]), l_svr.coef_[0])
plt.title('feature importances')
plt.xlabel('feature')
plt.ylabel('weight')

In [ ]:
print("coef_ shape:", l_svr.coef_.shape)
print("correlation_measure kind:", correlation_measure.kind)
print("X_train shape:", X_train.shape)

In [ ]:
print(X_train.shape)         # combien de features entrent dans le modèle ?
print(l_svr.coef_.shape)     # ce que le SVC a appris
# Si X_train a 276 colonnes → ton atlas n'est pas 100 ROIs au moment du fit
print(correlation_measure.mean_.shape)

In [ ]:
correlation_measure.inverse_transform(l_svr.coef_).shape

In [ ]:
from nilearn import plotting

feat_exp_matrix=correlation_measure.inverse_transform(l_svr.coef_)[0]

In [ ]:
plot_matrix = plotting.plot_matrix(feat_exp_matrix, figure = (10, 8),labels = range(feat_exp_matrix.shape[0]), reorder = False, tri = 'lower')

In [ ]:
# Overall connectomes
coords = plotting.find_parcellation_cut_coords(atlas_filename)

In [ ]:
plotting.plot_connectome(feat_exp_matrix, coords, colorbar = True)

In [ ]:
plotting.plot_matrix(feat_exp_matrix, figure = (10, 8),
                     labels = range(feat_exp_matrix.shape[0]),
                     reorder = False,
                     tri = 'lower',
                     vmin=0.001,
                     vmax=0.0015,)

In [ ]:
# Set the threshold for the displayed nodes
plotting.plot_connectome(feat_exp_matrix, coords, colorbar = True, edge_threshold = 0.0015)

donc très laid et on voit presque rien on doit faire un connectogramme

le code était pas assez détaillé donc j'ai essayer de trouver un autre package plus facile

https://mne.tools/mne-connectivity/dev/generated/mne_connectivity.viz.plot_connectivity_circle.html
https://mne.tools/mne-connectivity/stable/auto_examples/mne_inverse_label_connectivity.html

In [ ]:
"""import matplotlib.pyplot as plt
import mne_connectivity
from mne_connectivity.viz import plot_connectivity_circle
from nilearn import datasets

node_names = [l.decode() if isinstance(l, bytes) else l
              for l in parcellations.labels]"""

# 2. Couleurs : gauche = bleu, droite = rouge
node_colors = ['steelblue' if n.startswith('LH') else 'tomato'
               for n in node_names]

In [ ]:
"""correlation_measure = ConnectivityMeasure(
    kind='correlation',
    vectorize=False,
    standardize='zscore_sample'
)

correlation_matrix = correlation_measure.fit_transform([time_series])[0]"""

In [ ]:
"""print(correlation_matrix.shape)
print(correlation_matrix.ndim)"""

In [ ]:
"""print(len(node_names))"""

In [ ]:
"""node_names = [
    l.decode() if isinstance(l, bytes) else l
    for l in parcellations.labels
]

# enlever le background
node_names = node_names[1:]"""

In [ ]:
"""print(len(node_names))  # doit être 100
print(correlation_matrix.shape)  # (100, 100)"""

In [ ]:
node_colors = node_colors[1:]

In [ ]:
"""print(np.min(correlation_matrix), np.max(correlation_matrix))"""

oups j'ai oublié d'enlevé les autos corrélations

In [ ]:
"""correlation_matrix_2 = correlation_matrix.copy()
np.fill_diagonal(correlation_matrix_2, 0)
print(np.min(correlation_matrix_2), np.max(correlation_matrix_2))"""

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(polar=True))

plot_connectivity_circle(
    correlation_matrix_2,
    node_names,
    n_lines=100,
    vmin=-1,
    vmax=1,
    node_colors=node_colors,
    fontsize_names=6,
    ax=ax,
    show=False
)

plt.show()

ajoutons une légende (https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.legend.html et https://matplotlib.org/stable/api/_as_gen/matplotlib.patches.Patch.html)

avec un peu de magie on peu arrivé à ca:

In [ ]:
# mapping couleur par réseau
region_colors = {
    'Vis': '#4477AA',
    'SomMot': '#66CCEE',
    'DorsAttn': '#228833',
    'SalVentAttn': '#CCBB44',
    'Limbic': '#EE6677',
    'Cont': '#AA3377',
    'Default': '#BBBBBB'
}

# couleurs directement à partir des noms
node_colors = [
    region_colors[name.split('_')[2]]
    for name in node_names
]

In [ ]:
import numpy as np

# Trier par réseau (index 2 comme tu fais déjà)
networks = [name.split('_')[2] for name in node_names]
sorted_indices = np.argsort(networks)

# Réordonner tout
node_names_sorted = [node_names[i] for i in sorted_indices]
node_colors_sorted = [node_colors[i] for i in sorted_indices]
correlation_matrix_sorted = correlation_matrix_2[np.ix_(sorted_indices, sorted_indices)]

In [ ]:
# Plotter
fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(polar=True))

plot_connectivity_circle(
    correlation_matrix_sorted,
    node_names_sorted,
    n_lines=100,
    vmin=-0.3606,
    vmax=0.8658,
    node_colors=node_colors_sorted,
    fontsize_names=0,
    textcolor='none',
    ax=ax,
    colormap='turbo',
    facecolor="white",
    node_edgecolor="#212A37",
    node_linewidth=1.5,
    show=False
)

handles = []
for region in region_colors:
    color = region_colors[region]
    patch = mpatches.Patch(color=color, label=region)
    handles.append(patch)

plt.legend(handles=handles, loc='upper left', bbox_to_anchor=(1.05, 1))

plt.show()

In [ ]:
networks_list = ['Vis', 'SomMot', 'DorsAttn', 'SalVentAttn', 'Limbic', 'Cont', 'Default']

for network in networks_list:
    # Indices des ROIs de ce réseau
    network_indices = [i for i, name in enumerate(node_names)
                      if name.split('_')[2] == network]

    # Matrice et coords de ce réseau seulement
    network_matrix = correlation_matrix_2[np.ix_(network_indices, network_indices)]
    network_coords = coords[network_indices]

    view = plotting.view_connectome(
    network_matrix,
    network_coords,
    edge_threshold='75%',
    node_size=10,
    title= network)

    view.save_as_html(f"connectome_{network}.html")
    display(view)

In [ ]:
# Pour chaque réseau, créer un connectome propre
networks_list = ['Vis', 'SomMot', 'DorsAttn', 'SalVentAttn', 'Limbic', 'Cont', 'Default']

for network in networks_list:
    # Indices des ROIs de ce réseau
    network_indices = [i for i, name in enumerate(node_names)
                      if name.split('_')[2] == network]

    # Matrice et coords de ce réseau seulement
    network_matrix = correlation_matrix_2[np.ix_(network_indices, network_indices)]
    network_coords = coords[network_indices]

    # Plot connectome
    display = plotting.plot_connectome(
        network_matrix,
        network_coords,
        edge_threshold='75%',
        node_color=region_colors[network],
        node_size=30,
        edge_cmap='turbo',
        title=network,
        display_mode='z',      # vue de dessus
        colorbar=False
    )


In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
from IPython.display import Image
Image('connectogramme.png')

super maintenant faisons un connectogramme juste pour le DMN

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from mne_connectivity.viz import plot_connectivity_circle

dmn_indices = [
    i for i, name in enumerate(node_names)
    if 'Default' in name
]

dmn_names  = [node_names[i]  for i in dmn_indices]
dmn_colors = [node_colors[i] for i in dmn_indices]
dmn_matrix = correlation_matrix_2[np.ix_(dmn_indices, dmn_indices)]

In [ ]:

dmn_indices = [i for i, name in enumerate(node_names) if 'Default' in name]
dmn_names   = [node_names[i] for i in dmn_indices]
dmn_colors  = [node_colors[i] for i in dmn_indices]
dmn_matrix  = correlation_matrix_2[np.ix_(dmn_indices, dmn_indices)]
dmn_names_short = ['_'.join(name.split('_')[3:]) for name in dmn_names]

In [ ]:
print(np.min(dmn_matrix), np.max(dmn_matrix))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(polar=True))
plot_connectivity_circle(
    dmn_matrix,
    dmn_names_short,
    n_lines=100,
    vmin=-0.1522,
    vmax=0.8312,
    node_colors=dmn_colors,
    fontsize_names=7,
    textcolor='black',
    ax=ax,
    colormap='turbo',
    facecolor='white',
    node_edgecolor='#212A37',
    node_linewidth=1.5,
    show=False
)
plt.show()

In [ ]:
# Nœuds d'intérêt spécifiques
roi_names_of_interest = [
    # TPJ
    '7Networks_RH_Default_Par_1',
    '7Networks_LH_Default_Par_1',
    '7Networks_LH_Default_Par_2',
    # PCC
    '7Networks_LH_Default_pCunPCC_1',
    '7Networks_LH_Default_pCunPCC_2',
    '7Networks_RH_Default_pCunPCC_1',
    '7Networks_RH_Default_pCunPCC_2',
    # dmPFC
    '7Networks_LH_Default_PFC_5',
    '7Networks_LH_Default_PFC_6',
    '7Networks_RH_Default_PFCdPFCm_1',
    '7Networks_RH_Default_PFCdPFCm_2',
    '7Networks_RH_Default_PFCdPFCm_3',
    # vmPFC
    '7Networks_LH_Default_PFC_1',
    '7Networks_LH_Default_PFC_2',
    '7Networks_RH_Default_PFCv_1',
    '7Networks_RH_Default_PFCv_2',
]

In [ ]:
roi_indices = [i for i, name in enumerate(node_names) if name in roi_names_of_interest]
roi_names   = [node_names[i]  for i in roi_indices]
roi_colors  = [node_colors[i] for i in roi_indices]
roi_matrix  = correlation_matrix_2[np.ix_(roi_indices, roi_indices)]
roi_names_short = ['_'.join(name.split('_')[3:]) for name in roi_names]

In [ ]:
print(np.min(roi_matrix), np.max(roi_matrix))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(polar=True))
plot_connectivity_circle(
    roi_matrix,
    roi_names_short,
    n_lines=100,
    vmin=-0.1006,
    vmax=0.8266,
    node_colors=roi_colors,
    fontsize_names=7,
    textcolor='black',
    ax=ax,
    colormap='turbo',
    facecolor='white',
    node_edgecolor='#212A37',
    node_linewidth=1.5,
    show=False
)
plt.show()